# DLinear Native Adapter v1

目标：把 `knowledge_backlog_v1` 中优先级最高的 `dlinear_mse` 从 proxy replacement 候选推进到 `src/model` 的 native sequence adapter。Notebook 只保留实验流程和中文结论；可复用训练、回测、analysis 逻辑都调用 `src/`。外部论文只保存 URL 和自有摘要，不复制全文。

In [1]:
from pathlib import Path
import json
import sys

REPO_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "AGENTS.md").exists() and (path / "src").exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data import build_supervised_dataset, load_soxl_support_frame
from src.model.capabilities import model_capability_table, model_fidelity_summary
from src.model.matrix import run_quantbench_matrix_smoke
from src.model.sequence import DLinearDecompositionRegressor
from src.model.training import fit_estimator_for_model

OUTPUT_DIR = REPO_ROOT / "research" / "dlinear_native_adapter_v1"
CARD_DIR = OUTPUT_DIR / "matrix_card"

outputs = run_quantbench_matrix_smoke(
    card_dir=CARD_DIR,
    start="2026-01-01",
    end="2026-06-09",
    train_end="2026-04-30",
    model_names=("ridge_mse", "dlinear_mse", "patch_tst_mse"),
)

frame = load_soxl_support_frame(start="2026-01-01", end="2026-06-09")
dataset = build_supervised_dataset(frame, horizon=1)
train = dataset.frame.loc[dataset.frame["date"] <= "2026-04-30"].copy()
estimator, constant_prediction = fit_estimator_for_model(
    "dlinear_mse",
    train=train,
    feature_columns=dataset.feature_columns[:5],
)
capabilities = model_capability_table(("ridge_mse", "dlinear_mse", "patch_tst_mse"))
dlinear_capability = capabilities.loc[capabilities["model"].eq("dlinear_mse")].iloc[0].to_dict()
comparison = outputs.model_comparison.to_dict(orient="records")
dlinear_row = next(row for row in comparison if row["model"] == "dlinear_mse")

summary = {
    "schema_version": "research.dlinear_native_adapter_v1",
    "status": "completed",
    "paper_reference": {
        "source_id": "arxiv:2205.13504",
        "url": "https://arxiv.org/abs/2205.13504",
        "usage": "自有摘要和结构假设；不复制论文全文。",
    },
    "model_names": ["ridge_mse", "dlinear_mse", "patch_tst_mse"],
    "dlinear_estimator": estimator.__class__.__name__,
    "dlinear_has_trend_head": hasattr(estimator, "trend_model_"),
    "dlinear_has_residual_head": hasattr(estimator, "residual_model_"),
    "constant_prediction": constant_prediction,
    "dlinear_paper_fidelity": dlinear_capability["paper_fidelity"],
    "dlinear_is_proxy": bool(dlinear_capability["is_proxy"]),
    "model_fidelity": model_fidelity_summary(("ridge_mse", "dlinear_mse", "patch_tst_mse")),
    "top_model": outputs.summary["top_model"],
    "dlinear_return_pct": float(dlinear_row["return_pct"]),
    "dlinear_relative_return_pct": float(dlinear_row["relative_return_pct"]),
    "dlinear_max_drawdown_pct": float(dlinear_row["max_drawdown_pct"]),
    "dlinear_sharpe": float(dlinear_row["sharpe"]),
    "analysis_summary": outputs.summary.get("analysis_summary", {}),
    "artifacts": {
        "matrix_card": "matrix_card/",
        "analysis_summary": "matrix_card/analysis_summary.json",
        "model_comparison": "matrix_card/model_comparison.csv",
        "model_capabilities": "matrix_card/model_capabilities.csv",
        "daily_backtest": "matrix_card/daily_backtest.csv",
        "model_run_ledger": "matrix_card/model_run_ledger.csv",
        "model_run_ledger_summary": "matrix_card/model_run_ledger_summary.json",
        "artifact_manifest": "matrix_card/artifact_manifest.json",
    },
}
(OUTPUT_DIR / "notebook_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
summary


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'schema_version': 'research.dlinear_native_adapter_v1',
 'status': 'completed',
 'paper_reference': {'source_id': 'arxiv:2205.13504',
  'url': 'https://arxiv.org/abs/2205.13504',
  'usage': '自有摘要和结构假设；不复制论文全文。'},
 'model_names': ['ridge_mse', 'dlinear_mse', 'patch_tst_mse'],
 'dlinear_estimator': 'DLinearDecompositionRegressor',
 'dlinear_has_trend_head': True,
 'dlinear_has_residual_head': True,
 'constant_prediction': None,
 'dlinear_paper_fidelity': 'native_sequence',
 'dlinear_is_proxy': False,
 'model_fidelity': {'schema_version': 'model.fidelity.v1',
  'model_count': 3,
  'fidelity_counts': {'native_library': 1,
   'native_sequence': 1,
   'proxy_feature_adapter': 1},
  'proxy_models': ['patch_tst_mse'],
  'native_models': ['dlinear_mse', 'ridge_mse'],
  'promotion_blocker_count': 2},
 'top_model': 'patch_tst_mse',
 'dlinear_return_pct': 21.933701268101903,
 'dlinear_relative_return_pct': -36.18467989584157,
 'dlinear_max_drawdown_pct': 13.778714952100103,
 'dlinear_sharpe': 2.9

## 中文结论

- 做了什么：在 `src/model.sequence` 增加显式 `DLinearDecompositionRegressor`，用移动平均拆出 trend/residual，两条线性 head 分别拟合后相加预测；`dlinear_mse` 训练路径已切到该 estimator，能力标记从 `proxy_feature_adapter` 升级为 `native_sequence`。
- 核心指标：本 notebook 训练并回测 `ridge_mse`、`dlinear_mse`、`patch_tst_mse`，输出 `model_comparison.csv`、`daily_backtest.csv`、`model_run_ledger.csv`、`qlib_risk_metrics.csv`、`model_effect_deltas.csv` 和 manifest。具体收益、回撤、Sharpe 以 `notebook_summary.json` 与 `matrix_card/analysis_summary.json` 为准。
- regime 差异：本轮复用 `src/analysis` 的 window/regime、qlib risk、model effect delta；DLinear 是否优于 ridge 或 PatchTST 不能只看全区间收益，还要看 stress/downtrend 下相对收益、回撤和稳定性。
- 是否晋升：否。`native_sequence` 表示实现形态已从 proxy 进步为稳定训练接口，但 production 晋升仍需要更长历史窗口、更多 regime、成本压力和 paper/sim replay 复验。
- 风险：当前是紧凑实现，不等价于完整论文训练规模；SOXL 单标的样本短，容易过拟合；外部论文仅作为结构参考，未复制第三方代码或全文。
- 下一步：继续按 backlog 验证 `patch_tst_mse`、`rsr_mse`、`thgnn_mse` 的 proxy replacement，并把 DLinear 放入后续全模型矩阵和 production paper replay 的候选池。